
#  OpenMP Programming Tutorial and Hands-on

# <font color="green"> Problem 1 :  Change the number of threads</font>
* Execute them with various numbers of threads and see what happens

_<font color="green">Answer for trivial work omitted</font>_

# <font color="green"> Problem 2 :  Executing multiple statements by threads</font>
* Change following program so that both "world" and "good bye" are printed as many times as the number of threads

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_hello_ans.c
#include <stdio.h>

int main() {
  printf("hello\n");
#pragma omp parallel
  {
    printf("world\n");
    printf("good bye\n");
  }
  return 0;
}

In [ ]:
clang -fopenmp omp_hello_ans.c -o omp_hello_ans
# nvc -mp omp_hello_ans.c -o omp_hello_ans

In [ ]:
OMP_NUM_THREADS=3 ./omp_hello_ans


# <font color="green"> Problem 3 :  Using `omp_get_num_threads()` and `omp_get_thread_num()`</font>
* Change following program so that each thread prints its id in the team and the number of threads in the team, like this.  The exact order of lines may differ.  Strictly speaking, even characters in two lines can be mixed into a single line.

```
hello
0/5 world
4/5 world
1/5 world
3/5 world
2/5 world
good bye
```

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_hello_id_ans.c
#include <stdio.h>
#include <omp.h>

int main() {
  printf("hello\n");
#pragma omp parallel
  {
    int omp_nthreads = omp_get_num_threads();
    int omp_rank = omp_get_thread_num();
    printf("world %d/%d\n", omp_rank, omp_nthreads);
  }
  printf("good bye\n");
  return 0;
}

In [ ]:
clang -fopenmp omp_hello_id_ans.c -o omp_hello_id_ans
# nvc -mp omp_hello_id_ans.c -o omp_hello_id_ans

In [ ]:
OMP_NUM_THREADS=3 ./omp_hello_id_ans

# <font color="green"> Problem 4 :  How does `#pragma omp for` divide iterations to threads?</font>
* Execute the following cell and observe which iteration is executed by which thread
* Based on the observation, change the number of iterations and threads and predict the mapping between iterations and threads

In [ ]:

%%writefile omp_for.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
#pragma omp parallel
  {
    printf("I am thread %d in a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
#pragma omp for
    for (int i = 0; i < 24; i++) {
      usleep(100 * 1000 * i);
      printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
      fflush(stdout);
    }
  }
  return 0;
}


In [ ]:

clang -fopenmp omp_for.c -o omp_for
# nvc -mp omp_for.c -o omp_for

In [ ]:

OMP_NUM_THREADS=4 ./omp_for

_<font color="green">Example answer:</font>_

observation: in the default scheduling policy, which seems static, when executing with $P$ threads, the first $1/P$ of all iterations are executed by thread 0, next $1/P$ iterations by thread 1, etc.

# <font color="green"> Problem 5 :  Specifying the scheduling policy by schedule clause</font>
* In the following (artificial) loop, iteration _i_ roughly sleeps for (100 x _i_) milliseconds and this is almost exactly the time it takes
1. reason about the executing time of the parallel for loop with the default (static) scheduling policy
1. vary the scheduling policy and reason about their execution times

* What should be the execution time with the default (static) policy?
* What should be the execution time with the dynamic policy?
* Compare them with what you observed 

_<font color="green">Example answer:</font>_

In this loop, later iterations take longer.  Therefore, 

* in the static policy, the execution time of the entire parallel loop will be the execution time of the thread 3, which are assigned iterations 18 - 23.  the execution time will therefore be ((18 + 19 + ... + 23) * 100) milliseconds = 12.3 seconds

* in the dynamic policy, a crude (and optimistic) estimation, which assumes a perfect load balancing, is simply the serial execution time divided by the number of threads, which is ((0 + 1 + ... + 23) * 100) / 4 milliseconds = 6.9 seconds.  To be more precise in this particular case, the likely scenario will be the following (a number represents the last digit of an iteration number).

```
[0] 44448888888822222222222266666666666666660000000000000000000
[1] 15555599999999933333333333337777777777777777711111111111111111111
[2] 22666666000000000044444444444444888888888888888888222222222222222222222
[3] 33377777771111111111155555555555555599999999999999999933333333333333333333333
```

In this case, the execution time will be (3 + 7 + ... + 23) * 100 milliseconds = 7.8 seconds

# <font color="green"> Problem 6 :  Apply collapse and schedule</font>
* Apply collapse and schedule to the following loop
* Trick: write `schedule(runtime)` and you can change the scheduling policy at execution time by setting environment variable `OMP_SCHEDULE=` in the command line. See [OMP_SCHEDULE environment variable](https://www.openmp.org/spec-html/5.0/openmpsu41.html#x64-1370002.9.2.1) for details
* Reason about the execution time of various schedule policies and with/without collapse

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_collapse_ans.c

#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
  /* apply collapse and schedule */
#pragma omp parallel for collapse(2) schedule(runtime)
  for (int i = 0; i < 5; i++) {
    for (int j = 0; j < 5; j++) {
      usleep(100 * 1000 * (i + j));
      printf("iteration (%d, %d) executed by thread %d\n", i, j, omp_get_thread_num());
      fflush(stdout);
    }
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}



In [ ]:
clang -fopenmp omp_collapse_ans.c -o omp_collapse_ans
# nvc -mp omp_collapse_ans.c -o omp_collapse_ans

In [ ]:
OMP_NUM_THREADS=3 ./omp_collapse_ans

# <font color="green"> Problem 7 :  A quiz about recursive tasks</font>
* Answer the following questions
* How many tasks are created by `recursive_tasks(n, 0)`?  Include the caller of `recursive_tasks(n, 0)` as a task.  i.e., consider `recursive_tasks(0, 0)` creates one task
* How many of them are leaf tasks?

_<font color="green">Example answer:</font>_

Let $T(n)$ be the number of times `recursive_tasks(n, 0)` encounters during the execution.  Then,

* $T(0) = 0$
* $T(n) = 2 + 2T(n - 1)$

Therefore, $T(n) = 2^{n+1} - 2$.  Counting the caller of `recursive_tasks(n, 0)` as a task, the answer is $(2^{n+1} - 1)$.

Let $L(n)$ be the number of leaf tasks created by `recursive_tasks(n, 0)`.  Then,

* $L(0) = 1$
* $L(n) = 2 L(n - 1)$

Therefore, the number of leaf tasks $= L(n) = 2^n$

* Approximately what is the ideal execution time of `recursive_tasks(5, 0)` when using 10 threads?
* Compare it with what you observed

_<font color="green">Example answer:</font>_

Considering a leaf task takes 300 milliseconds, the time for non-leaf tasks will be negligible.
$2^5 = 32$ tasks will be distributed among 10 threads, so assuming a good load balancing, two of the ten threads will execute four leaf tasks and the other eight threads three leaf tasks.  Therefore execution time will be $300 \times 4$ milliseconds = 1.2 seconds.

_<font color="green">Example answer:</font>_

* any algorithm for which recursive calls is useful and those recursive calls can be run in parallel will benefit from task parallelism. to name a few,

  * quicksort
  * mergesort
  * fast fourier transform
  * matrix matrix multiplication (formulated with divide-and-conquer for locality)
  * sparse matrix-vector multiplication (formulated with divide-and-conquer for locality)
  * $N$-body simulation
  * KD-tree construction


# <font color="green"> Problem 8 :  Observe the effect of privatization</font>
* Execute the following and make sense of the output
* Add the `private(x)` clause and observe the difference

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_private_ans.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
  /* add private(x) clause below and see the difference */
#pragma omp parallel private(x)
  {
    int id = omp_get_thread_num();
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


# <font color="green"> Problem 9 :  Observe the effect of `private` and `firstprivate`</font>
* Execute the following and observe the output
* Add the `private(x)` clause and execute it
* Add the `firstprivate(x)` clause and execute it
* Make sense of the differences

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_firstprivate_ans.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
  /* add private(x)/firstprivate(x) clause and see the difference */
#pragma omp parallel private(x)
  {
    int id = omp_get_thread_num();
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


In [ ]:
%%writefile omp_firstprivate_ans.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
  /* add private(x)/firstprivate(x) clause and see the difference */
#pragma omp parallel firstprivate(x)
  {
    int id = omp_get_thread_num();
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


# <font color="green"> Problem 10 :  Apply `#pragma omp critical`</font>
* Execute the following program a few times and observe the result
* Add `#pragma omp critical` to the statement `x++` and observe the result

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_critical_ans.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel
  {
    int id = omp_get_thread_num();
#pragma omp critical
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


In [ ]:
clang -fopenmp omp_critical_ans.c -o omp_critical_ans
# nvc -mp omp_critical_ans.c -o omp_critical_ans

In [ ]:
OMP_NUM_THREADS=10 ./omp_critical_ans

# <font color="green"> Problem 11 :  Apply `#pragma omp atomic`</font>
* Add `#pragma omp atomic` to the statement `x++` and observe the result

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_atomic_ans.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel
  {
    int id = omp_get_thread_num();
#pragma omp atomic
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


In [ ]:
clang -fopenmp omp_atomic_ans.c -o omp_atomic_ans
# nvc -mp omp_atomic_ans.c -o omp_atomic_ans

In [ ]:
OMP_NUM_THREADS=10 ./omp_atomic_ans

# <font color="green"> Problem 12 :  Apply reduction</font>
* Add `reduction` clause to `#pragma omp parallel` below and observe the result

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_reduction_ans.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
#pragma omp parallel reduction(+:x)
  {
    int id = omp_get_thread_num();
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}


In [ ]:
clang -fopenmp omp_reduction_ans.c -o omp_reduction_ans
# nvc -mp omp_reduction_ans.c -o omp_reduction_ans

In [ ]:
OMP_NUM_THREADS=10 ./omp_reduction_ans

## 17-1. Apply a user-defined reduction
* Here is a simple (broken) parallel for loop that is meant to do a reduction on 3-element vector
* Define a reduction with `#pragma omp declare reduction` and apply it to the parallel loop

In [ ]:
%%writefile omp_ud_reduction_ans.c
#include <stdio.h>
#include <unistd.h>
#include <math.h>
#include <omp.h>

/* 3-element vector */
typedef struct {
  double a[3];
} vec_t;

/* x += y */
void vec_add(vec_t * x, vec_t * y) {
  for (int i = 0; i < 3; i++) {
    x->a[i] += y->a[i];
  }
}

/* x = {0,0,0} */
void vec_init(vec_t * x) {
  for (int i = 0; i < 3; i++) {
    x->a[i] = 0;
  }
}

#pragma omp declare reduction                   \
  (vp : vec_t : vec_add(&omp_out,&omp_in))      \
  initializer(vec_init(&omp_priv))

/* add an appropriate #pragma omp declare reduction ... here */
  
int main() {
  vec_t v;
  vec_init(&v);
  double t0 = omp_get_wtime();
  /* add an appropriate reduction clause, so that
     the result is always {10000,10000,10000} */
#pragma omp parallel for reduction(vp:v)
  for (int i = 0; i < 30000; i++) {
    v.a[i % 3]++;
  }
  double t1 = omp_get_wtime();
  printf("ans = {%.1f, %.1f, %.1f} in %f sec\n", v.a[0], v.a[1], v.a[2], t1 - t0);
  return 0;
}


In [ ]:
clang -fopenmp omp_ud_reduction_ans.c -o omp_ud_reduction_ans
# nvc -mp omp_ud_reduction_ans.c -o omp_ud_reduction_ans

In [ ]:
OMP_NUM_THREADS=10 ./omp_ud_reduction_ans

# <font color="green"> Problem 13 :  Putting them together: calculating an integral</font>
Write an OpenMP program that calculates

$$ \int \int_D \sqrt{1 - x^2 - y^2}\,dx\,dy $$

where

$$ D = \{\;(x, y)\;|\;0\leq x \leq 1, 0\leq y \leq 1, x^2 + y^2 \leq 1 \}$$

* Note: an alternative way to put it is to calculate

$$ \int_0^1 \int_0^1 f(x)\,dx\,dy $$

where

$$ f(x) = \left\{\begin{array}{ll}\sqrt{1 - x^2 - y^2} & (x^2 + y^2 \leq 1) \\ 0 & (\mbox{otherwise}) \end{array}\right. $$

* Use a nested loop to calculate the double integral
* Try work-sharing for, taskloop, recursive tasks to parallelize it
* The result should be close to $\pi/6 = 0.52359..$ (1/8 of the volume of the unit ball)
* Play with the number of infinitesimal intervals for integration and the number of threads so that you can observe a speedup
* As you are using a shared cloud environment, you do not have to be serious about speedup (nearly perfect speedup is unlikely when other students are simultaneously using the same machine and/or the cloud is doing many other stuff (e.g., servicing the page you are looking at right now)

* If you want to work with an editor you are accustomed to rather than web browser, see the `SSH (Remote Login)` section in `pd00_intro.sos` 

_<font color="green">Example answer:</font>_

In [ ]:
%%writefile omp_integral_ans.c
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

/* serial version */
double int_sqrt_one_minus_x2_y2(long n) {
  double h = 1.0 / n;
  double s = 0.0;
#pragma omp parallel for reduction(+:s) schedule(dynamic)
  for (long i = 0; i < n; i++) {
    for (long j = 0; j < n; j++) {
      double x = i * h ;
      double y = j * h;
      double z = 1 - x * x - y * y;
      if (z > 0.0) {
        s += sqrt(z);
      } else {
        break;
      }
    }
  }
  return s * h * h;
}

int main(int argc, char ** argv) {
  int i = 1;
  long n      = (argc > i ? atof(argv[i]) : 30L * 1000L); i++;
  double s = int_sqrt_one_minus_x2_y2(n);
  printf("s = %.9f (err = %e)\n", s, fabs(s - M_PI/6));
  return 0;
}

_<font color="green">Remarks:</font>_

* The first question is whether or not you use `collapse` to parallelize the inner loop
* If you do, using the break statement in the inner loop becomes invalid, so you cannot break the inner loop even when $(1 - x * x - y * y)$ becomes $< 0$
* If you do not, it is safe to do so, and in fact you want to do so as soon as $(1 - x * x - y * y)$ becomes $< 0$, to avoid wasting time calculating $(1 - x * x - y * y)$ in later iterations when you know they are all negative
* If the outer loop has enough iterations ($\gg$ the number of threads), then the latter is much more efficient

* Either case, the right scheduling strategy will be dynamic, as the work per iteration is not uniform.  If you do collapse, the work per (inner) iteration differs depending on whether you perform `sqrt` or not.  If you do not collapse, the work per (outer) iteration differs more significantly depending on where you break the loop.
* If you do collapse, the pitfall is that the work per iteration becomes so small that using the default chunk size (1) makes the dynamic load balancing overhead too large to enjoy any benefit of parallelizaiton. Make the chunk size larger to amortize the overhead. If you do not collapse, a single iteration executes an entire inner loop, so the overhead of dynamic load balancing is hardly an issue in practice.

In [ ]:
clang -fopenmp omp_integral_ans.c -o omp_integral_ans
# nvc -mp omp_integral.c -o omp_integral

In [ ]:
OMP_NUM_THREADS=8 ./omp_integral_ans